# 03 — Modelagem dos Dados | Camada Gold

Neste notebook é construída a camada Gold do pipeline.
A tabela Silver consolidada anteriormente possui informações de país, ano, sexo, indicador nutricional e prevalência. Para facilitar as consultas analíticas, os dados serão organizados em um modelo dimensional no formato Star Schema.
O modelo será composto por uma tabela fato central e quatro dimensões: país, tempo, sexo e indicador nutricional.

## 1. Leitura da camada Silver

A tabela consolidada da camada Silver é carregada como fonte para a criação do modelo dimensional.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_nutrition = spark.table(
    "workspace.default.silver_nutrition"
)

display(silver_nutrition.limit(20))

## 2. Definição da granularidade

A tabela fato terá como granularidade uma observação de prevalência para cada combinação de:

- país;
- ano;
- sexo;
- indicador nutricional.

A medida principal da tabela fato será a prevalência percentual (`prevalence_pct`).

## 3. Dimensão País

A dimensão de país contém as localidades presentes no conjunto de dados e seus respectivos códigos.

Também é criada uma chave numérica para utilização no relacionamento com a tabela fato.

In [0]:
window_country = Window.orderBy("country", "country_code")

dim_country = (
    silver_nutrition
    .select("country", "country_code")
    .distinct()
    .withColumn(
        "country_id",
        F.row_number().over(window_country)
    )
    .select(
        "country_id",
        "country",
        "country_code"
    )
)

display(dim_country.limit(20))

## 4. Dimensão Tempo

A dimensão de tempo representa os anos disponíveis na base.
Além do ano, foi criada uma chave numérica que será utilizada pela tabela fato.

In [0]:
window_time = Window.orderBy("year")

dim_time = (
    silver_nutrition
    .select("year")
    .distinct()
    .withColumn(
        "time_id",
        F.row_number().over(window_time)
    )
    .select(
        "time_id",
        "year"
    )
)

display(dim_time)

## 5. Dimensão Sexo

A dimensão de sexo representa as duas categorias existentes nos conjuntos analisados.

In [0]:
window_sex = Window.orderBy("sex")

dim_sex = (
    silver_nutrition
    .select("sex")
    .distinct()
    .withColumn(
        "sex_id",
        F.row_number().over(window_sex)
    )
    .select(
        "sex_id",
        "sex"
    )
)

display(dim_sex)

## 6. Dimensão Indicador

A dimensão de indicador identifica qual condição nutricional está sendo analisada.
Neste projeto são utilizados os indicadores de obesidade e excesso de peso.

In [0]:
window_indicator = Window.orderBy("indicator")

dim_indicator = (
    silver_nutrition
    .select("indicator")
    .distinct()
    .withColumn(
        "indicator_id",
        F.row_number().over(window_indicator)
    )
    .select(
        "indicator_id",
        "indicator"
    )
)

display(dim_indicator)

## 7. Construção da tabela fato

A tabela fato é construída relacionando os registros da camada Silver às dimensões criadas anteriormente.
Os atributos descritivos são substituídos pelas respectivas chaves das dimensões, mantendo como medida principal a prevalência percentual.

In [0]:
fact_nutrition = (
    silver_nutrition

    .join(
        dim_country,
        ["country", "country_code"],
        "left"
    )

    .join(
        dim_time,
        ["year"],
        "left"
    )

    .join(
        dim_sex,
        ["sex"],
        "left"
    )

    .join(
        dim_indicator,
        ["indicator"],
        "left"
    )

    .select(
        "country_id",
        "time_id",
        "sex_id",
        "indicator_id",
        "prevalence_pct"
    )
)

display(fact_nutrition.limit(20))

## 8. Validação da tabela fato

Após a construção da tabela fato, é verificado se a quantidade de registros permaneceu igual à encontrada na camada Silver.
Essa validação ajuda a identificar perdas ou multiplicações de registros causadas por relacionamentos incorretos durante os joins.

In [0]:
print(
    "Registros na Silver:",
    silver_nutrition.count()
)

print(
    "Registros na tabela fato:",
    fact_nutrition.count()
)

### Verificação das chaves dimensionais

Também é verificado se algum registro da tabela fato ficou sem correspondência nas dimensões criadas.

In [0]:
display(
    fact_nutrition.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in [
            "country_id",
            "time_id",
            "sex_id",
            "indicator_id",
            "prevalence_pct"
        ]
    ])
)

## 9. Persistência da camada Gold

Após a construção e validação do modelo dimensional, as dimensões e a tabela fato são persistidas em formato Delta.
Essas tabelas representam a camada Gold do pipeline e serão utilizadas para responder às perguntas de análise definidas no início do projeto.

In [0]:
dim_country.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_dim_country")

dim_time.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_dim_time")

dim_sex.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_dim_sex")

dim_indicator.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_dim_indicator")

fact_nutrition.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_fact_nutrition")

## 10. Verificação das tabelas Gold

Por fim, as tabelas disponíveis no catálogo são consultadas para confirmar a persistência do modelo dimensional.

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.default
    """)
)

## 11. Teste do modelo dimensional

Como verificação final, é realizada uma consulta relacionando a tabela fato às dimensões. O objetivo é reconstruir uma visão analítica legível a partir das chaves do modelo dimensional.

In [0]:
teste_modelo = (
    fact_nutrition

    .join(dim_country, "country_id")
    .join(dim_time, "time_id")
    .join(dim_sex, "sex_id")
    .join(dim_indicator, "indicator_id")

    .select(
        "country",
        "country_code",
        "year",
        "sex",
        "indicator",
        "prevalence_pct"
    )

    .filter(F.col("country") == "Brazil")

    .orderBy(
        "year",
        "indicator",
        "sex"
    )
)

display(teste_modelo.limit(20))

## Conclusão

A camada Gold foi construída utilizando um modelo dimensional em esquema estrela.
O modelo é composto por uma tabela fato contendo os valores de prevalência e pelas dimensões de país, tempo, sexo e indicador nutricional.
Após a criação das tabelas, foram realizadas verificações para garantir que nenhum registro fosse perdido durante os relacionamentos e que todas as chaves dimensionais fossem preenchidas corretamente.
Com a camada Gold concluída, os dados estão preparados para a etapa final do projeto: a análise dos indicadores nutricionais e a resposta às perguntas definidas no objetivo.